In [4]:
import sys
import logging
from typing import Dict, Any, List, Optional, Annotated
from typing_extensions import TypedDict
import uuid

# 导入 LangGraph 核心组件
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import AIMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool

logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("PolicyGateHITL")

# ============================================================================
# 1. 具备幂等性与动态账本校验的 Mock 后端
# ============================================================================

class MockBankingBackend:
    def __init__(self, initial_balance: float = 5000.0):
        self.balance = initial_balance
        self.debit_count = 0
        self.ledger: Dict[str, dict] = {} # 幂等键 -> 执行结果

    def get_balance(self) -> float:
        return self.balance

    def execute_transfer(self, amount: float, recipient: str, idempotency_key: str) -> dict:
        # 【幂等防线】如果该幂等键已处理过，直接返回历史结果，绝不重复扣款！
        if idempotency_key in self.ledger:
            logger.info(f"🛡️ [Idempotency Protection] 捕获到重复键: {idempotency_key}，直接返回历史结果，不重复扣款。")
            return self.ledger[idempotency_key]

        # 实时余额校验
        if amount > self.balance:
            return {"success": False, "reason": "INSUFFICIENT_FUNDS"}

        self.balance -= amount
        self.debit_count += 1
        
        result = {
            "success": True, 
            "remaining_balance": self.balance, 
            "transferred": amount,
            "debit_count": self.debit_count
        }
        self.ledger[idempotency_key] = result
        return result

# 全局后端实例
backend = MockBankingBackend(initial_balance=5000.0)

@tool
def get_balance() -> dict:
    """查询当前账户余额"""
    logger.info("⚙️ [Tool Execution] 执行只读工具: get_balance()")
    return {"balance": backend.get_balance()}

@tool
def execute_transfer(amount: float, recipient: str, idempotency_key: str) -> dict:
    """执行高危副作用转账工具"""
    logger.warning(f"🚨 [Tool Execution] 执行转账: amount={amount}, recipient={recipient}, key={idempotency_key}")
    return backend.execute_transfer(amount, recipient, idempotency_key)

tools = [get_balance, execute_transfer]
tool_node_executor = ToolNode(tools)

# ============================================================================
# 2. 状态定义 (AgentState)
# ============================================================================

class PendingCallInfo(TypedDict):
    tool_call_id: str
    tool_name: str
    args: Dict[str, Any]

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    pending_call: Optional[PendingCallInfo]      # 当前等待审批的高危调用
    approval_result: Optional[Dict[str, Any]]    # 恢复后收到的审批数据
    filtered_ai_message: Optional[AIMessage]     # 供 ToolNode 专用的过滤消息

# ============================================================================
# 3. 图节点定义 (Nodes with interrupt & revalidation)
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    """1. LLM 决策节点：生成复合 Tool Calls"""
    logger.info("🤖 [agent_node] LLM 正在生成 Tool Calls...")
    ai_msg = AIMessage(
        content="请帮我查询余额并转账 1000 元给 Alice。",
        tool_calls=[
            {"id": "call_bal_01", "name": "get_balance", "args": {}},
            {"id": "call_trans_01", "name": "execute_transfer", "args": {
                "amount": 1000.0, 
                "recipient": "Alice", 
                "idempotency_key": "txn-key-9988"
            }}
        ]
    )
    return {"messages": [ai_msg]}


def policy_gate_node(state: AgentState) -> AgentState:
    """2. Policy Gate 节点：只读工具直接放行，副作用工具暂存为 pending_call"""
    logger.info("🛡️ [policy_gate_node] 审查 Tool Calls...")
    last_message = [m for m in state["messages"] if isinstance(m, AIMessage)][-1]
    
    allowed_calls = []
    pending = None
    
    for tc in last_message.tool_calls:
        if tc["name"] == "get_balance":
            allowed_calls.append(tc)
        elif tc["name"] == "execute_transfer":
            # 高危副作用工具需要中断审批，暂存
            pending = {
                "tool_call_id": tc["id"],
                "tool_name": tc["name"],
                "args": tc["args"]
            }
            
    filtered_ai = AIMessage(content=last_message.content, tool_calls=allowed_calls)
    return {
        "filtered_ai_message": filtered_ai,
        "pending_call": pending
    }


def approval_node(state: AgentState) -> AgentState:
    """
    3. 审批与中断节点 (Human-in-the-Loop)：
    调用内置 interrupt() 暂停图的执行，向外抛出结构化请求并等待 resume。
    """
    logger.info("⏸️ [approval_node] 触发 interrupt()，等待人类审批输入...")
    p_call = state["pending_call"]
    
    approval_data = interrupt({
        "tool_call_id": p_call["tool_call_id"],
        "tool_name": p_call["tool_name"],
        "args": p_call["args"],
        "question": "是否批准这笔高危转账？"
    })
    
    logger.info(f"▶️ [approval_node] 收到恢复数据 (resume): {approval_data}")
    return {"approval_result": approval_data}


def revalidate_and_execute_node(state: AgentState) -> AgentState:
    """
    4. 恢复后的重新校验与执行节点：
    - 检查审批结果（是否批准、身份、参数是否被篡改）。
    - 恢复后实时重新检查余额，绝不使用暂停前的旧余额。
    - 合格后交由 ToolNode 真实执行。
    """
    logger.info("🔍 [revalidate_and_execute_node] 收到恢复信号，开始安全重新校验...")
    approval = state.get("approval_result", {})
    p_call = state.get("pending_call")
    
    # 解析恢复后的审批状态
    is_approved = approval.get("approved", False)
    if not is_approved:
        logger.warning("🛑 [Revalidation Failed] 用户拒绝了该笔审批，流程终止，扣款 0 次。")
        return {"messages": [ToolMessage(content="ERROR: USER_REJECTED_APPROVAL", tool_call_id=p_call["tool_call_id"])]}

    # 动态重新检查余额
    current_balance = backend.get_balance()
    required_amount = p_call["args"]["amount"]
    if required_amount > current_balance:
        logger.warning(f"🛑 [Revalidation Failed] 暂停期间余额降至 {current_balance}，不足以支付 {required_amount}，取消转账！")
        return {"messages": [ToolMessage(content="ERROR: CANCELLED_DUE_TO_BALANCE_DROPPED_DURING_PAUSE", tool_call_id=p_call["tool_call_id"])]}

    # 校验全部通过，执行转账工具
    logger.info("✅ [Revalidation Passed] 重新校验全部通过，执行转账工具...")
    args = p_call["args"]
    res = backend.execute_transfer(
        amount=args["amount"], 
        recipient=args["recipient"], 
        idempotency_key=args["idempotency_key"]
    )
    
    tool_msg = ToolMessage(content=str(res), tool_call_id=p_call["tool_call_id"])
    return {"messages": [tool_msg]}


def tool_execution_router(state: AgentState) -> str:
    """条件路由 1：判断是否有只读工具需要先执行"""
    filtered_ai = state.get("filtered_ai_message")
    if filtered_ai and len(filtered_ai.tool_calls) > 0:
        return "run_readonly_tools"
    return "go_to_approval"


def readonly_tool_executor_node(state: AgentState) -> AgentState:
    """执行只读工具（如 get_balance）"""
    logger.info("⚡ [readonly_tool_executor_node] 执行只读工具...")
    filtered_ai = state["filtered_ai_message"]
    execution_state = {"messages": [filtered_ai]}
    result = tool_node_executor.invoke(execution_state)
    return {"messages": result["messages"]}

# ============================================================================
# 4. 构建带 Checkpointer 的状态图
# ============================================================================

def build_hitl_graph():
    builder = StateGraph(AgentState)
    
    builder.add_node("agent", agent_node)
    builder.add_node("policy_gate", policy_gate_node)
    builder.add_node("readonly_tool_node", readonly_tool_executor_node)
    builder.add_node("approval_node", approval_node)
    builder.add_node("revalidate_node", revalidate_and_execute_node)
    
    # 边与条件路由
    builder.add_edge(START, "agent")
    builder.add_edge("agent", "policy_gate")
    
    builder.add_conditional_edges(
        "policy_gate",
        tool_execution_router,
        {
            "run_readonly_tools": "readonly_tool_node",
            "go_to_approval": "approval_node"
        }
    )
    
    builder.add_edge("readonly_tool_node", "approval_node")
    builder.add_edge("approval_node", "revalidate_node")
    builder.add_edge("revalidate_node", END)
    
    return builder.compile(checkpointer=InMemorySaver())

# ============================================================================
# 5. 测试用例与验收验证 (Verification & Test Cases)
# ============================================================================

def run_day38_verification_tests():
    print("=" * 70)
    print("🧪 Day 38: HITL Interrupt & Checkpointer 恢复验证测试")
    print("=" * 70)

    # ------------------------------------------------------------------------
    # 测试场景 1：余额 5000，转账 1000，未审批 -> 预期暂停，扣款 0 次
    # ------------------------------------------------------------------------
    print("\n--- [Test 1] 运行至 interrupt 暂停并验证 Checkpoint 状态 ---")
    global backend
    backend = MockBankingBackend(initial_balance=5000.0)
    
    graph = build_hitl_graph()
    thread_config = {"configurable": {"thread_id": "transfer-thread-001"}}
    
    graph.invoke({"messages": []}, config=thread_config)
    
    snapshot = graph.get_state(thread_config)
    print(f"当前图暂停所在的节点 (next nodes): {snapshot.next}")
    print(f"当前扣款次数: {backend.debit_count}")
    
    assert "approval_node" in snapshot.next, "图必须准确停留在 approval_node 节点等待恢复"
    assert backend.debit_count == 0, "暂停期间绝不能发生扣款"
    print("✅ 测试 1 通过：图成功暂停，扣款 0 次。")

    # ------------------------------------------------------------------------
    # 测试场景 2：用户批准，但暂停期间余额降到 800 -> 恢复后取消，扣款 0 次
    # ------------------------------------------------------------------------
    print("\n--- [Test 2] 暂停期间余额突降，恢复后触发重新校验取消 ---")
    backend.balance = 800.0 
    print(f"模拟并发：暂停期间账户余额降至: {backend.balance}")

    resume_command = Command(resume={
        "approved": True,
        "tool_call_id": "call_trans_01",
        "args": {"amount": 1000.0, "recipient": "Alice", "idempotency_key": "txn-key-9988"}
    })
    
    graph.invoke(resume_command, config=thread_config)
    
    print(f"恢复并执行完毕后的最终账户余额: {backend.balance}")
    print(f"实际扣款次数 (debit_count): {backend.debit_count}")
    
    assert backend.debit_count == 0, "余额不足时重新校验必须拦截，扣款必须为 0"
    print("✅ 测试 2 通过：恢复后动态重新校验成功拦截，扣款 0 次。")

    # ------------------------------------------------------------------------
    # 测试场景 3：恢复且余额充足 (5000)，正常批准并测试幂等不重复扣款
    # ------------------------------------------------------------------------
    print("\n--- [Test 3] 正常批准通过转账，并测试重复恢复/重试的幂等性 ---")
    backend = MockBankingBackend(initial_balance=5000.0)
    thread_config_2 = {"configurable": {"thread_id": "transfer-thread-002"}}
    
    # 1. 运行至暂停
    graph.invoke({"messages": []}, config=thread_config_2)
    
    # 统一测试用的幂等键（必须与 agent_node 中生成的一致）
    target_idempotency_key = "txn-key-9988"

    # 2. 正常批准恢复
    resume_command_ok = Command(resume={
        "approved": True,
        "tool_call_id": "call_trans_01",
        "args": {"amount": 1000.0, "recipient": "Alice", "idempotency_key": target_idempotency_key}
    })
    graph.invoke(resume_command_ok, config=thread_config_2)
    
    print(f"正常转账成功后的余额: {backend.balance}, 扣款次数: {backend.debit_count}")
    assert backend.debit_count == 1, "应当成功扣款 1 次"

    # 3. 模拟使用相同的 Idempotency Key 再次尝试重复执行（测试幂等保护）
    print(">>> 模拟相同幂等键再次尝试重复执行...")
    backend.execute_transfer(amount=1000.0, recipient="Alice", idempotency_key=target_idempotency_key)
    
    print(f"再次执行后的最终余额: {backend.balance}, 最终扣款次数: {backend.debit_count}")
    assert backend.debit_count == 1, "幂等键生效，重复执行绝对不能二次扣款！"
    
    print("\n🎉 Day 38 所有验收测试全部满分通过：真正实现了 Interrupt 暂停、Checkpointer 恢复、动态重新校验与幂等扣款防线！")

if __name__ == "__main__":
    run_day38_verification_tests()

2026-10-20 16:09:52,148 - [INFO] - 🤖 [agent_node] LLM 正在生成 Tool Calls...
2026-10-20 16:09:52,151 - [INFO] - 🛡️ [policy_gate_node] 审查 Tool Calls...
2026-10-20 16:09:52,153 - [INFO] - ⚡ [readonly_tool_executor_node] 执行只读工具...
2026-10-20 16:09:52,158 - [INFO] - ⚙️ [Tool Execution] 执行只读工具: get_balance()
2026-10-20 16:09:52,164 - [INFO] - ⏸️ [approval_node] 触发 interrupt()，等待人类审批输入...
2026-10-20 16:09:52,172 - [INFO] - ⏸️ [approval_node] 触发 interrupt()，等待人类审批输入...
2026-10-20 16:09:52,174 - [INFO] - ▶️ [approval_node] 收到恢复数据 (resume): {'approved': True, 'tool_call_id': 'call_trans_01', 'args': {'amount': 1000.0, 'recipient': 'Alice', 'idempotency_key': 'txn-key-9988'}}
2026-10-20 16:09:52,177 - [INFO] - 🔍 [revalidate_and_execute_node] 收到恢复信号，开始安全重新校验...
2026-10-20 16:09:52,178 - [WARNING] - 🛑 [Revalidation Failed] 暂停期间余额降至 800.0，不足以支付 1000.0，取消转账！
2026-10-20 16:09:52,183 - [INFO] - 🤖 [agent_node] LLM 正在生成 Tool Calls...
2026-10-20 16:09:52,187 - [INFO] - 🛡️ [policy_gate_node] 审查 Tool Calls...


🧪 Day 38: HITL Interrupt & Checkpointer 恢复验证测试

--- [Test 1] 运行至 interrupt 暂停并验证 Checkpoint 状态 ---
当前图暂停所在的节点 (next nodes): ('approval_node',)
当前扣款次数: 0
✅ 测试 1 通过：图成功暂停，扣款 0 次。

--- [Test 2] 暂停期间余额突降，恢复后触发重新校验取消 ---
模拟并发：暂停期间账户余额降至: 800.0
恢复并执行完毕后的最终账户余额: 800.0
实际扣款次数 (debit_count): 0
✅ 测试 2 通过：恢复后动态重新校验成功拦截，扣款 0 次。

--- [Test 3] 正常批准通过转账，并测试重复恢复/重试的幂等性 ---
正常转账成功后的余额: 4000.0, 扣款次数: 1
>>> 模拟相同幂等键再次尝试重复执行...
再次执行后的最终余额: 4000.0, 最终扣款次数: 1

🎉 Day 38 所有验收测试全部满分通过：真正实现了 Interrupt 暂停、Checkpointer 恢复、动态重新校验与幂等扣款防线！
